# Corbett et al germline variants
Corbett et al. have done germline annotations of the original PBTA dataset. Even though it's not the full dataset, are there associations to be found?

In [ ]:
import pandas as pd
import scipy
import statsmodels
import sys
sys.path.append('../src')
from data_imports import *

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

In [ ]:
# data imports
def import_corbett_patients(path='data/Corbett_2025_suppl_tbls/TableS2_cohort-summary.xlsx'):
    return pd.read_excel(path,skiprows=1,index_col=0)
def import_corbett_snvs(path='data/Corbett_2025_suppl_tbls/TableS5_syndrome_variants.xlsx'):
    return pd.read_excel(path,skiprows=1,index_col=0,sheet_name='S5A.SNVs-InDels')
def import_corbett_svs(path='data/Corbett_2025_suppl_tbls/TableS5_syndrome_variants.xlsx'):
    return pd.read_excel(path,skiprows=1,index_col=0,sheet_name='S5B.structural-variants')
patients = import_patients()
c_patients = import_corbett_patients()
c_snvs = import_corbett_snvs()
c_svs = import_corbett_svs()

In [ ]:
# data integration
def generate_germline_dataset(patients,c_patients,c_snvs,c_svs):
    # integrate all corbett annotations
    s = pd.concat([c_snvs['Gene symbol'].copy(),c_svs['Gene symbol'].copy()])
    s = s.groupby(level=0).agg(set)
    df = patients[patients.index.isin(c_patients.index)].copy()
    df['germline_variants'] = s
    df['germline_variants'] = df.germline_variants.map(lambda x: set() if pd.isna(x) else x)
    df['germline_syndrome'] = ~df.germline_variants.isna()
    return df
def filter_germline_dataset(df):
    tumortypes = set(df[df.amplicon_class == 'ecDNA'].cancer_type.unique())
    print(f'Relevant tumor types: {tumortypes}')
    df = df[df.cancer_type.isin(tumortypes)]
    return df

In [ ]:
df = generate_germline_dataset(patients,c_patients,c_snvs,c_svs)
print(len(df[df.germline_syndrome]))
print(len(df))
df = filter_germline_dataset(df)
print(len(df[df.germline_syndrome]))
print(len(df))
df.head()

In [ ]:
def count(df):
    '''
    Count germline variants by cancer type and amplicon class.
    '''
    vars_of_interest = ['cancer_type','amplicon_class','germline_syndrome']
    df = df[vars_of_interest].copy()
    df['count']=1
    return df.groupby(vars_of_interest).count()
count(df)

In [ ]:
# What gene variants present in this dataset?
sorted(df.germline_variants.explode().dropna().unique())

In [ ]:
def test_gene(df,gene):
    ct = pd.crosstab(df.amplicon_class,df.germline_variants.map(lambda x: gene in x))
    
    # test enrichment in amplifications
    ct = pd.crosstab(df.amplicon_class != 'no amplification',df.germline_variants.map(lambda x: gene in x))
    print(f'Testing enrichment of {gene} in amplifications...')
    print(ct)
    print(scipy.stats.chi2_contingency(ct))
    
    # test enrichment in ecDNA
    df = df[df.amplicon_class != 'no amplification']
    ct = pd.crosstab(df.amplicon_class != 'intrachromosomal',df.germline_variants.map(lambda x: gene in x))
    print(f'Testing enrichment of {gene} in ecDNA...')
    print(ct)
    print(scipy.stats.chi2_contingency(ct))
    return

def test_all_genes(df):
    for gene in df.germline_variants.explode().dropna().unique():
        test_gene(df,gene)
test_all_genes(df)

# TP53 the only significant result. Woefully underpowered though.
'''
Testing enrichment of TP53 in amplifications...
germline_variants  False  True 
amplicon_class                 
False                353      0
True                  79      8
Chi2ContingencyResult(statistic=np.float64(28.110394755407945), pvalue=np.float64(1.145887454059067e-07), dof=1, expected_freq=array([[346.58181818,   6.41818182],
       [ 85.41818182,   1.58181818]]))

Testing enrichment of TP53 in ecDNA...
germline_variants  False  True 
amplicon_class                 
False                 33      2
True                  46      6
Chi2ContingencyResult(statistic=np.float64(0.2954546442481568), pvalue=np.float64(0.5867460831763742), dof=1, expected_freq=array([[31.7816092,  3.2183908],
       [47.2183908,  4.7816092]]))
'''

In [ ]:
df[df.germline_variants.map(lambda x: 'ELP1' in x)]